## Actividad 3_19: Arroz
<div style="border-style:groove;border-width:thin;padding:10px">
En esta actividad vamos a utilizar las técnicas de redes neuronales y deep learning que hemos visto en clase para solucionar el problema de clasificar tipos de arroz.
    <ol>
        <li>Descarga el archivo "Rice_Image_Dataset.zip" de <a href="https://www.muratkoklu.com/datasets/vtdhnd09.php">https://www.muratkoklu.com/datasets/vtdhnd09.php</a>. Descomprime el archivo y guarda la carpeta en un lugar adecuado.</li>
        <li>Importa los datos usando las misma técnica que en la actividad de los pistachos (En este caso, 250x250 en escala de grises).</li>
        <li>Guarda las etiquetas de los datos. Debes tener un conjunto photos y otro labels que estén ordenados igual. labels debe tener números entre 0.0 y 4.0, ya que hay 5 clases de arroz.</li>
        <li>Utiliza PCA para reducir el dataset.</li>
        <li>Soluciona el ejercicio con una red neuronal. Puedes utilizar todas las técnicas que hemos aprendido.</li>
        <li>Soluciona el ejercicio usando una red convolucional.</li>
    </ol>
</div>

In [3]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
import pandas as pd
import numpy as np
#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('./Rice_Image_Dataset/')
#Clase Kirmizi será la clase 0.0
#Clase Siirt será la clase 1.0


photos =  []
labels = []

In [4]:
#2. IMPORTAMOS LOS DATOS:
for idx,folder in enumerate(folders):
    for file in listdir('./Rice_Image_Dataset/'+folder):
        #Cargamos la imagen.
        #load_img sirve para cargar las imágenes en memoria. Tiene distintos parámetros para modificar como se cargan las imágenes.
        photo = load_img('./Rice_Image_Dataset/'+folder+'/' + file, target_size=(128, 128), color_mode="grayscale") 
        #Convertimos la imagen a un array.
        photo = img_to_array(photo)
        #Los guardamos en las listas.
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print(idx)

0
1
2
3
4


In [5]:
photos = np.array(photos)
labels = np.array(labels)

photos = photos.astype('float32')

### PCA

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Normalizar
X = photos / 255.0
y = labels

# 2. Creamos la versión aplanada para poder aplicar PCA
X_flat = X.reshape(len(X), -1)

# 3. Aplicamos PCA sobre la versión aplanada
pca = PCA(n_components=100, random_state=42)
X_pca = pca.fit_transform(X_flat)

# 4. Dividimos los datos para la Red Neuronal Normal (ANN) usando PCA
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)

# 5. Dividimos los datos para la Red Convolucional (CNN) usando las imágenes originales
X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(X, y, test_size=0.2, random_state=42)

### Aplicando la red neuronal

In [7]:
# Haciendo la red neuronal a partir del tratamiento de PCA
from tensorflow import keras

model = keras.Sequential()
model.add(keras.layers.Flatten(input_shape=(X_pca.shape[1],)))
# Luego metemos capas ocultas
model.add(keras.layers.Dense(800, activation="relu"))
model.add(keras.layers.Dense(400, activation="relu"))
# Luego metemos la capa de salida, que tiene 5 neuronas, una por cada clase, y función de activación softmax, que es la que se suele usar para clasificación multiclase.
model.add(keras.layers.Dense(5, activation="softmax"))

from tensorflow.keras import optimizers
sgd = optimizers.SGD(learning_rate=0.0005)

model.compile(loss="sparse_categorical_crossentropy", optimizer=sgd, metrics=["accuracy"])

/home/ciabd10/anaconda3/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
W0000 00:00:1776181320.770866   24378 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [8]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/100
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8486 - loss: 0.5824 - val_accuracy: 0.9297 - val_loss: 0.3165
Epoch 2/100
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9421 - loss: 0.2559 - val_accuracy: 0.9468 - val_loss: 0.2128
Epoch 3/100
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9528 - loss: 0.1901 - val_accuracy: 0.9557 - val_loss: 0.1694
Epoch 4/100
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9590 - loss: 0.1579 - val_accuracy: 0.9612 - val_loss: 0.1447
Epoch 5/100
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9632 - loss: 0.1382 - val_accuracy: 0.9647 - val_loss: 0.1288
Epoch 6/100
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9661 - loss: 0.1249 - val_accuracy: 0.9667 - val_loss: 0.1174
Epoch 7/100
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9679 - loss: 0.1154 - val_accuracy: 0.9688 - val_loss: 0.1088
Epoch 8/100
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9695 - loss: 0

In [9]:
model.evaluate(X_test, y_test)

469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 898us/step - accuracy: 0.9780 - loss: 0.0638


[0.06380672007799149, 0.9779999852180481]

### Ahora aplicaremos una red convolucional

In [10]:
# Definición de la red convolucional
model_cnn = keras.Sequential()

# Capa de convolución: el input_shape debe ser (128, 128, 1)
model_cnn.add(keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(128, 128, 1)))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

model_cnn.add(keras.layers.Flatten())
model_cnn.add(keras.layers.Dense(64, activation="relu"))
model_cnn.add(keras.layers.Dense(5, activation="softmax"))

# Optimizador y compilación
sgd_cnn = optimizers.SGD(learning_rate=0.0005)
model_cnn.compile(optimizer=sgd_cnn, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

/home/ciabd10/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=300, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/30
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 64s 38ms/step - accuracy: 0.9447 - loss: 0.2092 - val_accuracy: 0.9535 - val_loss: 0.1734
Epoch 2/30
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 64s 38ms/step - accuracy: 0.9560 - loss: 0.1552 - val_accuracy: 0.9592 - val_loss: 0.1372
Epoch 3/30
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 64s 38ms/step - accuracy: 0.9617 - loss: 0.1307 - val_accuracy: 0.9653 - val_loss: 0.1173
Epoch 4/30
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 64s 38ms/step - accuracy: 0.9651 - loss: 0.1169 - val_accuracy: 0.9672 - val_loss: 0.1086
Epoch 5/30
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 65s 38ms/step - accuracy: 0.9664 - loss: 0.1083 - val_accuracy: 0.9677 - val_loss: 0.1026
Epoch 6/30
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 65s 38ms/step - accuracy: 0.9682 - loss: 0.1026 - val_accuracy: 0.9705 - val_loss: 0.0925
Epoch 7/30
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 65s 38ms/step - accuracy: 0.9690 - loss: 0.0982 - val_accuracy: 0.9717 - val_loss: 0.0890
Epoch 8/30
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 65s 38ms/step - accuracy: 0.9699 -

In [13]:
model_cnn.evaluate(X_test_cnn_4d, y_test_cnn)

469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9713 - loss: 0.0795


[0.07952626049518585, 0.9712666869163513]